<a href="https://colab.research.google.com/github/abhisheknaik2k20/SMA_Project/blob/main/SMA_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import/Install Packages**

In [1]:
!pip install -q supabase
!pip install -q google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 2.7 MB/s eta 0:00:00


In [3]:
import re
import json
import time
import random
import gspread
import requests
import pandas as pd
import requests as req
from google import genai
from datetime import datetime
from bs4 import BeautifulSoup
from google.colab import userdata, auth
from google.colab import userdata as env
from supabase import create_client, Client
from googleapiclient.discovery import build
from google.oauth2.service_account import Credentials

# **POSTGRE DATABASE API CODE**

In [ ]:
class PostgreSQLDatabase:
  def __init__(self):
    try:
      self.supabase_client : Client = create_client(env.get('supabase_URL'), env.get('supabase_key'))
    except Exception as error : print(error)

  def populate_genre(self, data=[]):
    try:
      response=self.supabase_client.table('Genre').insert(data, count='exact').execute()
      return response.data, response.count
    except Exception as error : print(error)

  def get_genre_info(self):
    try:
      response=self.supabase_client.table('Genre').select('*', count ='exact').execute()
      return response.data, response.count
    except Exception as error : print(error)

  def populate_video(self, data=[]):
    try:
      response = self.supabase_client.table('Video').insert(data, count='exact').execute()
      return response.data, response.count
    except Exception as error : print(error)

  def get_video_info(self):
    try:
      response=self.supabase_client.table('Video').select('*', count ='exact').execute()
      return response.data, response.count
    except Exception as error : print(error)

  def return_video_ids(self):
    try:
      response= self.supabase_client.table('Video').select('video_id').execute()
      return [r['video_id'] for r in response.data]
    except Exception as error : print(error)

  def populate_comments(self, data=[]):
    try:
      response = self.supabase_client.table('Comments').insert(data, count='exact').execute()
      return response.data, response.count
    except Exception as error : print(error)

  def fetch_comments_data(self):
    try:
      response=self.supabase_client.table('Comments').select('*').execute()
      return response.data
    except Exception as error : print(error)

  def return_comment_ids(self):
    try:
      response= self.supabase_client.table('Comments').select('comment_id').execute()
      return [r['comment_id'] for r in response.data]
    except Exception as error : print(error)

  def return_video_ids_with_no_comment_data(self):
    try:
      video_ids = set(self.return_video_ids())
      commented_video_ids = set(r['video_id'] for r in self.supabase_client.rpc('get_distinct_comment_video_ids').execute().data)
      return list(video_ids.difference(commented_video_ids))
    except Exception as error : print(error)

  def get_channel_ids(self):
    try:
      channel_ids = [r['channel_id'] for r in self.supabase_client.rpc('get_distinct_channel_ids').execute().data]
      return channel_ids
    except Exception as error : print(error)

  def populate_channel(self, channel_data=[]):
    try : return self.supabase_client.table('Channel').upsert(channel_data,on_conflict='channel_id',ignore_duplicates=True,count='exact').execute()
    except Exception as error : print(error)

  def populate_subreddit(self, subreddits = []):
    try : return self.supabase_client.table('Subreddits').insert(subreddits, count='exact').execute()
    except Exception as error : print(error)

  def populate_posts(self, posts = []):
    try : return self.supabase_client.table('Posts').insert(posts, count='exact').execute()
    except Exception as error : print(error)

  def insert_redditor(self, redditor_data):
    try:
        result = self.supabase_client.table('Redditor').upsert(redditor_data).execute()
        return result
    except Exception as e:
        print(f"Error inserting redditor {redditor_data.get('name')}: {e}")
        return None

# **Reddit API Code**

In [ ]:
class FetchRedditData:
  def __init__(self):
    self.url = env.get('reddit_API_URL')
    self.headers = {'x-rapidapi-key': env.get('reddit_API_key'),'x-rapidapi-host': env.get('reddit_API_host')}

  def make_api_call(self ,url=env.get('reddit_API_URL'), endpoint='', query_params={}):
    try:
      response = req.get(f"{url}/{endpoint}", headers=self.headers, params=query_params)
      return response.json()
    except Exception as error : print(error)

  def return_popular_subreddits(self):
    response=self.make_api_call(endpoint='getPopularSubreddits')
    if response['success'] :
      print(response)
      result=[]
      for sub_data in response['data']['subreddits']:
        data = sub_data['data']
        result.append({'id' : data.get('id'),'name' : data.get('display_name'),'name_prefixed' : data.get('display_name_prefixed'),'title' : data.get('title'),'url' : data.get('url'),'icon_url' : data.get('icon_img'),'banner_url' : data.get('banner_img'),'header_url' : data.get('header_img'),'subscribers' : data.get('subscribers'),'public_description' : data.get('public_description'),'description_html' : data.get('description_html'),'description' : data.get('description'),'created_utc' : datetime.fromtimestamp(data.get('created_utc')).isoformat(),'primary_color' : data.get('primary_color'),'key_color' : data.get('key_color'),'banner_background_color' : data.get('banner_background_color'),'advertiser_category' : data.get('advertiser_category')})
      return result

  def store_popular_subreddits(self):
    pgdb = PostgreSQLDatabase()
    subreddits = self.return_popular_subreddits()
    if not subreddits : return
    pgdb.populate_subreddit(subreddits=subreddits)

  def return_popular_posts_from_subreddit(self, query_params={}):
    return self.make_api_call(endpoint='getTopPostsBySubreddit', query_params= query_params)

  def store_popular_posts_from_subreddit(self, ):
    pgdb = PostgreSQLDatabase()
    subreddits = pgdb.supabase_client.table('Subreddits').select('*').execute().data
    for subreddit in subreddits:
      response = self.return_popular_posts_from_subreddit(query_params= {"subreddit":subreddit['name'],"time":"year"})
      result = []
      for sub_data in response['data']['posts']:
        data = sub_data['data']
        result.append({"id": data.get('id'),"name": data.get('name'),"permalink": data.get('permalink'),"title": data.get('title'),"author": data.get('author'),"subreddit_id": data.get('subreddit_id')[3:],"created_utc": datetime.fromtimestamp(data.get('created_utc', 0)).isoformat(),"score": data.get('score'),"ups": data.get('ups'),"upvote_ratio": data.get('upvote_ratio'),"num_comments": data.get('num_comments'),"num_crossposts": data.get('num_crossposts'),"total_awards_received": data.get('total_awards_received'),"domain": data.get('domain'),"url": data.get('url'),"post_hint": data.get('post_hint'),"is_video": data.get('is_video'),"is_gallery": data.get('is_gallery'),"thumbnail_url": data.get('thumbnail'),"preview" : data.get('preview')})
      pgdb.populate_posts(result)

**Fetching and Storing Reddit Data**

In [ ]:
reddit = FetchRedditData()
reddit.store_redditor_data()

Rate limited for Mightymaas. Retry 1/5 in 60s
Rate limited for Far_Pay8487. Retry 1/5 in 60s
Rate limited for firstyasmine. Retry 1/5 in 60s
Rate limited for JeffreyHugh. Retry 1/5 in 60s
Rate limited for DeniseDv. Retry 1/5 in 60s
Rate limited for Ihatemisinfo. Retry 1/5 in 60s
API error for [deleted]: Invalid input, name - [deleted]
Rate limited for Soul_Sleepwhale. Retry 1/5 in 60s
Rate limited for Chad-Cat. Retry 1/5 in 60s
Rate limited for damaniac1223. Retry 1/5 in 60s
Rate limited for Cafritsz. Retry 1/5 in 60s
Rate limited for TopConcentrate8484. Retry 1/5 in 60s
Rate limited for grayfox0430. Retry 1/5 in 60s
Rate limited for Crushedfire. Retry 1/5 in 60s
Rate limited for kootles10. Retry 1/5 in 60s
Rate limited for newsspotter. Retry 1/5 in 60s
Rate limited for Zyxtriann. Retry 1/5 in 60s
Rate limited for GenDouglasMacArthur. Retry 1/5 in 60s
CSV saved: redditors_20260223_114903.csv
Error storing redditors batch: {'message': 'ON CONFLICT DO UPDATE command cannot affect row a s

# **YOUTUBE API CODE**

In [ ]:
class FetchYoutubeData:
  def __init__(self):
    try : self.youtube = build("youtube", "v3", developerKey=env.get('YT_v3') )
    except Exception as error :
      print(error)

  def fetch_video_ids(self, videoCategoryTag : int, order = 'viewCount'):
    try:
      search_response = self.youtube.search().list(part="id",type="video",videoCategoryId=videoCategoryTag, q=" ", order=order,relevanceLanguage="en",maxResults=20).execute()
      return [item["id"]["videoId"] for item in search_response["items"]]
    except Exception as error :
      print(error)

  def fetch_video_data(self,genre_id,video_ids):
    video_data=[]
    for video_id in video_ids:
      try:
        video_request = self.youtube.videos().list(part="snippet,contentDetails,statistics,status,topicDetails", id =  video_id).execute()
        item=video_request["items"][0]
        video_data.append({'video_id': item['id'],'title': item['snippet']['title'],'description': item['snippet']['description'],'published_at': item['snippet']['publishedAt'],'channel_id': item['snippet']['channelId'],'channel_title':item['snippet']['channelTitle'],'comment_count': int(item['statistics'].get('commentCount', 0)),'like_count': int(item['statistics'].get('likeCount', 0)),'view_count': int(item['statistics'].get('viewCount', 0)),'genre_id': genre_id})
      except Exception as error : continue
    self.fetch_channel_data(channel_ids= [r['channel_id'] for r in video_data])
    return video_data

  def store_video_data(self):
    pgdb = PostgreSQLDatabase()
    responses, count = pgdb.get_genre_info()
    if count == -1 : return
    stored_video_data = set(pgdb.return_video_ids())
    for response in responses:
        video_ids = self.fetch_video_ids(response['video_tag'], order = 'relevance')
        if not video_ids : continue
        result = set(video_ids).difference(stored_video_data)
        if not result: continue
        stored_video_data.update(result)
        pgdb.populate_video(self.fetch_video_data(response['genre_id'], result))

  def fetch_comment_data(self, video_id):
    result=[]
    try:
      request = self.youtube.commentThreads().list(part="snippet",videoId=video_id,textFormat="plainText", order="relevance", maxResults=10).execute()
      for item in request.get("items", []):
        top_comment = item["snippet"]["topLevelComment"]["snippet"]
        result.append({"comment_id": item["id"],"video_id": video_id,"comment_text": top_comment["textDisplay"],"like_count": str(top_comment["likeCount"]),"comment_date": top_comment["publishedAt"][:10],"channel_id": top_comment.get("authorChannelId", {}).get("value", "N/A"),})
      return result
    except Exception as error : return []

  def fetch_channel_data(self, channel_ids=[]):
    pgdb = PostgreSQLDatabase()
    existing_channel_data = set(pgdb.get_channel_ids())
    channel_data = []
    for channel_id in channel_ids:
        if channel_id in existing_channel_data : continue
        try:
            response = self.youtube.channels().list(part="snippet,statistics,contentDetails,localizations,topicDetails",id=channel_id).execute()
            item = response['items'][0]
            channel_data.append({'channel_id': item['id'],'channel_title': item['snippet']['title'],'description': item['snippet']['description'],'published_at': item['snippet']['publishedAt'],'country': item['snippet'].get('country', None),'default_language': item['snippet'].get('defaultLanguage', None),'view_count': int(item['statistics'].get('viewCount', 0)),'subscriber_count': int(item['statistics'].get('subscriberCount', 0)),'video_count': int(item['statistics'].get('videoCount', 0)),'topic_categories': item.get('topicDetails', {}).get('topicCategories', [])})
            existing_channel_data.add(item['id'])
        except Exception as error:continue
    if channel_data:pgdb.populate_channel(channel_data=channel_data)

  def store_comment_data(self):
     pgdb = PostgreSQLDatabase()
     video_ids = pgdb.return_video_ids_with_no_comment_data()
     for video_id in video_ids:
      comments = self.fetch_comment_data(video_id)
      if not comments : continue
      channel_ids =  [r['channel_id'] for r in comments]
      self.fetch_channel_data(channel_ids)
      pgdb.populate_comments(comments)

**Fetching and Storing Youtube Video Data**

In [ ]:
youtube = FetchYoutubeData()
youtube.store_video_data()
youtube.store_comment_data()

# **Google Sheets API Code**

In [7]:
class GoogleSheetData:
    def __init__(self, table_name ='', sheet_name =''):
      self.table_name = table_name
      self.sheet_name = sheet_name
      service_account_info = json.loads(userdata.get('google_service_account_json'))
      self.supabase_client = create_client(userdata.get('supabase_URL'),userdata.get('supabase_key'))
      creds = Credentials.from_service_account_info(service_account_info, scopes=["https://www.googleapis.com/auth/spreadsheets","https://www.googleapis.com/auth/drive"])
      self.google_sheets = gspread.authorize(creds)

    def fix_data_formatting(self):
      self.records.columns = ['timestamp', 'q1_age_range', 'q2_education_level', 'q3_digital_literacy','q6_daily_social_media_hours', 'q7_social_media_platforms','q8_has_youtube_account', 'q9_youtube_content_type','q10_uses_youtube_personalization', 'q11_subscribed_youtube_premium','q12_concern_data_use', 'q13_trust_platforms','q14_perceived_control', 'q15_reviews_privacy_settings','q16_privacy_gap', 'q17_biggest_barrier','q18_app_permissions_scenario', 'q19_conversation_tracking_scenario','q20_youtube_ad_profiling_reaction', 'q21_privacy_policy_email_behaviour','q22_ad_blocker_scenario']
      self.records['timestamp'] = pd.to_datetime(self.records['timestamp'], format='%d/%m/%Y %H:%M:%S').dt.strftime('%Y-%m-%dT%H:%M:%S')

    def access_gsheet(self):
      try:
        self.records = pd.DataFrame(self.google_sheets.open(self.sheet_name).sheet1.get_all_records())
        self.fix_data_formatting()
      except Exception as e:
        self.records = pd.DataFrame()
        print(f"Error: {e}")

    def fetch_data_from_table(self):
      self.database_records = pd.DataFrame(self.supabase_client.table(self.table_name).select('*').execute().data)
      self.database_records['timestamp'] = pd.to_datetime(self.database_records['timestamp']).dt.strftime('%Y-%m-%dT%H:%M:%S')

    def sync_to_supabase(self):
      merged = self.records.merge(self.database_records['timestamp'],on='timestamp',how='left',indicator=True)
      new_rows = merged[merged['_merge'] == 'left_only'].drop(columns='_merge')
      records_list = new_rows.to_dict(orient='records')
      if len(records_list) == 0 : return
      try : self.supabase_client.table(self.table_name).upsert(records_list).execute()
      except Exception as e : print(e)

**Fetching Info from GSheets and Saving in DataBase**

In [8]:
gsheet_data = GoogleSheetData(table_name ='Privacy Perception Study',sheet_name='Privacy Perception Study (Responses)')
gsheet_data.access_gsheet()
gsheet_data.fetch_data_from_table()
gsheet_data.sync_to_supabase()